In [ ]:
import numpy as np
import pandas as pd
import os
from scipy.interpolate import interp1d

# Paths
ESP32_DATA_PATH = "/Users/aleynagulkazdal/Desktop/esp32_data"
WIAR_DATA_PATH = "../data"

# Parametreler
TARGET_FRAMES = 300
TARGET_SC = 90

# Aktivite mapping 
ACTIVITY_MAP = {
    "horizontal_arm_wave": 0,   # WIAR 01
    "two_hands_wave": 2,        # WIAR 03
    "forward_kick": 7,          # WIAR 08
    "bend": 9,                  # WIAR 10
    "hand_clap": 10,            # WIAR 11
    "walk": 11,                 # WIAR 12
    "sit_down": 14,             # WIAR 15
    "squat": 15,                # WIAR 16
    "still": 16                 # Yeni sınıf WIAR'da yok, ESP32'de "still" olarak adlandırıldı
}

# Kullanıcı mapping 
USER_MAP = {
    "aleyna": 8,
    "damla": 9,
    "deniz": 10,
    "derya": 11
}

print("Importlar tamam!")
print(f"Aktivite sayısı: {len(ACTIVITY_MAP)}")
print(f"Kullanıcı sayısı: {len(USER_MAP)}")

Importlar tamam!
Aktivite sayısı: 9
Kullanıcı sayısı: 4


In [15]:
def parse_esp32_csv(filepath):
    # Tüm satırları tek tek oku, hatalıları atla
    rows = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    header = lines[0].strip().split(',')
    expected_cols = len(header)
    
    for line in lines[1:]:
        vals = line.strip().split(',')
        if len(vals) == expected_cols:
            try:
                rows.append([float(v) for v in vals])
            except:
                continue
    
    if len(rows) == 0:
        raise ValueError("Hiç satır yüklenemedi!")
    
    raw = np.array(rows)
    
    # sc_ sütunları → time hariç
    raw = raw[:, 1:]  # time sütununu at
    
    # İlk 4 değer header, at
    raw = raw[:, 4:]
    
    # Real + Imag → Amplitude
    real = raw[:, 0::2]
    imag = raw[:, 1::2]
    
    if real.shape[1] != imag.shape[1]:
        min_sc = min(real.shape[1], imag.shape[1])
        real = real[:, :min_sc]
        imag = imag[:, :min_sc]
    
    amp = np.sqrt(real**2 + imag**2)
    
    # → 90 SC interpolate
    x_old = np.linspace(0, 1, amp.shape[1])
    x_new = np.linspace(0, 1, TARGET_SC)
    
    interpolated = np.zeros((amp.shape[0], TARGET_SC))
    for i in range(amp.shape[0]):
        f = interp1d(x_old, amp[i], kind='linear')
        interpolated[i] = f(x_new)
    
    return interpolated

def fix_length(data, target_len=TARGET_FRAMES):
    current_len = data.shape[0]
    if current_len == target_len:
        return data
    
    x_old = np.linspace(0, 1, current_len)
    x_new = np.linspace(0, 1, target_len)
    
    # Her bir subcarrier (90 tane) için zaman ekseninde interpolasyon
    f = interp1d(x_old, data, axis=0, kind='linear', fill_value="extrapolate")
    return f(x_new)

In [16]:
X_esp32 = []
y_act_esp32 = []
y_user_esp32 = []

skipped = 0
loaded = 0

for person, user_label in USER_MAP.items():
    person_path = os.path.join(ESP32_DATA_PATH, person)
    if not os.path.exists(person_path):
        print(f"UYARI: {person} klasörü bulunamadı!")
        continue
    
    for activity, act_label in ACTIVITY_MAP.items():
        act_path = os.path.join(person_path, activity)
        if not os.path.exists(act_path):
            print(f"UYARI: {person}/{activity} klasörü bulunamadı!")
            continue
        
        csv_files = [f for f in os.listdir(act_path) if f.endswith(".csv")]
        
        for csv_file in sorted(csv_files):
            filepath = os.path.join(act_path, csv_file)
            try:
                x = parse_esp32_csv(filepath)
                x = fix_length(x)
                
                if x.shape != (300, 90):
                    skipped += 1
                    continue
                
                X_esp32.append(x)
                y_act_esp32.append(act_label)
                y_user_esp32.append(user_label)
                loaded += 1
                
            except Exception as e:
                print(f"HATA: {csv_file} → {e}")
                skipped += 1

X_esp32 = np.array(X_esp32)
y_act_esp32 = np.array(y_act_esp32)
y_user_esp32 = np.array(y_user_esp32)

print(f"Yüklenen: {loaded} | Atlanan: {skipped}")
print(f"X shape: {X_esp32.shape}")
print(f"Aktivite dağılımı: {np.bincount(y_act_esp32)}")
print(f"Kullanıcı dağılımı: {np.bincount(y_user_esp32)}")

/opt/anaconda3/envs/csi_env/lib/python3.10/site-packages/scipy/interpolate/_interpolate.py:479: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


Yüklenen: 720 | Atlanan: 0
X shape: (720, 300, 90)
Aktivite dağılımı: [80  0 80  0  0  0  0 80  0 80 80 80  0  0 80 80 80]
Kullanıcı dağılımı: [  0   0   0   0   0   0   0   0 180 180 180 180]


In [18]:
# NaN kontrolü
nan_count = np.isnan(X_esp32).sum()
print(f"NaN sayısı: {nan_count}")
print(f"NaN olan örnek sayısı: {np.isnan(X_esp32).any(axis=(1,2)).sum()}")

# NaN'ları 0 ile doldur
X_esp32_clean = np.nan_to_num(X_esp32, nan=0.0)

print(f"\nTemizleme sonrası:")
print(f"Min: {X_esp32_clean.min():.2f} | Max: {X_esp32_clean.max():.2f} | Mean: {X_esp32_clean.mean():.2f}")

NaN sayısı: 243000
NaN olan örnek sayısı: 9

Temizleme sonrası:
Min: -0.00 | Max: 90.19 | Mean: 17.11


In [19]:
# NaN olan örnekleri çıkar
nan_mask = ~np.isnan(X_esp32).any(axis=(1,2))
X_esp32_clean = X_esp32[nan_mask]
y_act_esp32_clean = y_act_esp32[nan_mask]
y_user_esp32_clean = y_user_esp32[nan_mask]

print(f"Temiz örnek sayısı: {len(X_esp32_clean)}")
print(f"Atılan NaN örnek: {(~nan_mask).sum()}")
print(f"Aktivite dağılımı: {np.bincount(y_act_esp32_clean)}")
print(f"Kullanıcı dağılımı: {np.bincount(y_user_esp32_clean)}")

Temiz örnek sayısı: 711
Atılan NaN örnek: 9
Aktivite dağılımı: [78  0 79  0  0  0  0 78  0 79 79 79  0  0 79 80 80]
Kullanıcı dağılımı: [  0   0   0   0   0   0   0   0 179 177 178 177]


In [22]:
print("WIAR kullanıcıları:", np.unique(y_user_wiar))
print("ESP32 kullanıcıları:", np.unique(y_user_esp32_clean))

# Çakışıyor mu?
overlap = set(np.unique(y_user_wiar)) & set(np.unique(y_user_esp32_clean))
print(f"Çakışan kullanıcılar: {overlap}")

WIAR kullanıcıları: [0 1 2 3 6 7 8 9]
ESP32 kullanıcıları: [ 8  9 10 11]
Çakışan kullanıcılar: {np.int64(8), np.int64(9)}
